# Prompt MCP Demo - Interactive Tutorial\n\nThis notebook demonstrates how to create and use an MCP server that encapsulates prompt engineering patterns for generating executive summaries and introductions from PDFs.\n\n## What You'll Learn\n\n- How MCP servers can embed prompt engineering expertise\n- a structured prompt format: Instruction + Context + Constraints\n- Automating report generation workflows\n\n## The Prompt Structure\n\n```\nInstruction:        [WHO + WHAT] You are a technical engineer...\nAdditional Context: [WHY + HOW] This follows industry guidelines...\nDocument Content:   [INPUT] Extracted PDF text...\nConstraints:        [BOUNDARIES] 5 bullets, 300 words max...\n```

## Step 1: Create MCP Project Folder\n\nFirst, we'll create a folder to hold our MCP server code.

In [ ]:
import os\n\n# Create mcp_project folder\nos.makedirs('mcp_project', exist_ok=True)\nprint("Created mcp_project/ folder")

## Step 2: Create the MCP Server\n\nWe'll use the `%%writefile` magic function to create the server file.

In [ ]:
%%writefile mcp_project/prompt_server.py\n\n\"\"\"\nMCP Server for PDF Report Generation - Prompt Engineering Demo\n\"\"\"\n\nimport json\nimport os\nfrom pathlib import Path\nfrom typing import Optional\nfrom mcp.server.fastmcp import FastMCP\n\n# PDF processing\ntry:\n    from docling.document_converter import DocumentConverter\n    DOCLING_AVAILABLE = True\nexcept ImportError:\n    DOCLING_AVAILABLE = False\n\nPDF_DIR = os.environ.get(\"PDF_DIR\", \"../pdf_files\")\nmcp = FastMCP(\"prompt-mcp-demo\")\n\n_converter = None\ndef get_converter():\n    global _converter\n    if _converter is None:\n        _converter = DocumentConverter()\n    return _converter\n\n# Prompt Templates\nTEMPLATES = {\n    \"executive_summary\": {\n        \"instruction\": \"You are a technical engineer. Write an executive summary...\",\n        \"context\": \"This follows industry guidelines for technical reports...\",\n        \"constraints\": \"- 5 bullets\n- Max 300 words\n- Professional tone\"\n    }\n}\n\n@mcp.tool()\ndef list_pdf_files(directory=None):\n    search_dir = directory or PDF_DIR\n    if not os.path.exists(search_dir):\n        return json.dumps({\"error\": f\"Directory not found: {search_dir}\"})\n    files = [f for f in os.listdir(search_dir) if f.endswith('.pdf')]\n    return json.dumps({\"files\": files, \"count\": len(files)})\n\n@mcp.tool()\ndef extract_pdf_content(file_path):\n    if not DOCLING_AVAILABLE:\n        return json.dumps({\"error\": \"docling not installed\"})\n    converter = get_converter()\n    result = converter.convert(file_path)\n    return json.dumps({\"content\": result.document.export_to_text()})\n\n@mcp.tool()\ndef generate_executive_summary(file_path):\n    template = TEMPLATES[\"executive_summary\"]\n    extraction = extract_pdf_content(file_path)\n    content = json.loads(extraction).get(\"content\", \"\")[:5000]\n    prompt = f\"{template['instruction']}\n\nContext: {template['context']}\n\nDocument: {content}\n\n{template['constraints']}\"\n    return json.dumps({\"prompt\": prompt})\n\nif __name__ == \"__main__\":\n    mcp.run(transport='stdio')\n

## Step 3: Create pyproject.toml\n\nDefine project dependencies.

In [ ]:
%%writefile mcp_project/pyproject.toml\n\n[project]\nname = \"prompt-mcp-demo\"\nversion = \"0.1.0\"\ndescription = \"MCP Server for PDF report generation\"\nrequires-python = \">=3.10\"\ndependencies = [\n    \"mcp>=1.26.0\",\n    \"docling>=2.0.0\",\n]\n

## Step 4: Set Up the Project\n\nRun these commands in your terminal:\n\n```bash\ncd mcp_project\nuv init\nuv venv\nsource .venv/bin/activate\nuv add mcp docling\n```

## Step 5: Launch MCP Inspector\n\n```bash\ncd mcp_project\nnpx @modelcontextprotocol/inspector uv run prompt_server.py\n```\n\nOpen the displayed URL in your browser.

## Step 6: Test the Tools\n\n### List PDF Files\n- **Tool**: `list_pdf_files`\n\n### Extract PDF Content\n- **Tool**: `extract_pdf_content`\n- **Parameters**: `file_path`: `../pdf_files/report.pdf`\n\n### Generate Executive Summary\n- **Tool**: `generate_executive_summary`\n- **Parameters**: `file_path`: `../pdf_files/report.pdf`\n\n## Understanding the Output\n\nThe `generate_executive_summary` tool returns a **constructed prompt** that follows the standard format:\n\n```\nYou are a technical engineer. Write an executive summary...\n\nContext: This follows industry guidelines...\n\nDocument: [Extracted PDF text here]\n\nConstraints: - 5 bullets, Max 300 words...\n```\n\n**Copy this prompt and send it to your LLM** (GPT-4, Claude, etc.) to get the executive summary!

## Key Teaching Points\n\n### Why This Pattern Works\n\n1. **Instruction** defines WHO (technical engineer) and WHAT (write summary)\n2. **Context** provides WHY (guidelines) and HOW (approach)\n3. **Constraints** set BOUNDARIES (format, length, tone)\n\n### MCP Value\n\n- **Encapsulates expertise**: prompt engineering patterns are encoded in the server\n- **Hides complexity**: PDF extraction is automatic\n- **Reusable**: Same templates work across projects\n\n### Workflow\n\nPDF Report → MCP Server → Constructed Prompt → LLM → Executive Summary

## Next Steps\n\n- Add your own PDF files to `pdf_files/` folder\n- Modify templates in `prompt_server.py`\n- Create new templates for other use cases\n- Experiment with different constraint formats